# SpaceX Falcon 9 - Interactive Visual Analytics with Folium
## IBM Applied Data Science Capstone

In [ ]:
!pip install folium
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df.head()

## Task 1 - Mark all launch sites on a map

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
launch_sites = spacex_df.groupby('Launch Site').first().reset_index()
for idx, row in launch_sites.iterrows():
    folium.CircleMarker(location=[row['Lat'], row['Long']], radius=10, color='#d35400', fill=True, fill_color='#d35400').add_to(site_map)
    folium.Marker(location=[row['Lat'], row['Long']], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)
site_map

## Task 2 - Mark success/failed launches for each site

In [ ]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x==1 else 'red')
for idx, row in spacex_df.iterrows():
    folium.Marker(location=[row['Lat'], row['Long']], icon=folium.Icon(color=row['marker_color']), popup=f"{row['Launch Site']} - Class: {row['class']}").add_to(marker_cluster)
site_map.add_child(marker_cluster)
site_map

## Task 3 - Calculate distance to proximities - coastline, highway, railway, city

In [ ]:
from math import sin, cos, sqrt, atan2, radians
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1 = radians(lat1); lon1 = radians(lon1); lat2 = radians(lat2); lon2 = radians(lon2)
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*atan2(sqrt(a), sqrt(1-a))
    return R*c
# Example: CCAFS SLC-40 to coastline
launch_site_lat = 28.563197; launch_site_lon = -80.576820
coastline_lat = 28.563197; coastline_lon = -80.567820
print('Distance to coastline:', calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon), 'km')
# Draw line
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
line = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(line)
folium.Marker(coordinates[1], icon=DivIcon(icon_size=(20,20), html='<div style="font-size:12px;color:#d35400;"><b>0.86 KM</b></div>')).add_to(site_map)
site_map

## Conclusion: Launch sites are near coast, highways and railways for logistics, far from cities for safety.
## Author - Nasri Prince